# T03 — MCP 클라이언트 기초 (Tutorial)

> 🎓 **학생 친화 튜토리얼** — low-level `ClientSession`부터 시작해서 Tool Use 루프 + 클래스 추상화까지 단계적으로 학습합니다.

## 학습 목표
1. **low-level** API (`ClientSession` + `stdio_client`)로 서버 연결
2. `list_tools()` / `call_tool()` 메서드 사용법
3. **Claude API + MCP 도구** 결합 — Tool Use 루프 패턴 재활용
4. 함수 → 클래스 추상화 단계 학습
5. Skilljar 원본 `MCPClient` 클래스로 자연스럽게 연결

## Prerequisites
- T01·T02 완료 (`tutorial_server.py` 존재 + Inspector로 동작 확인)
- `ANTHROPIC_API_KEY` 환경 변수 (또는 `.env` 파일)
- 약 35분 소요

> 📖 **강의노트 매핑**: `Week_07.md §1.2` (Client Architecture), `§2.1` (Implementing Client). Skilljar 원본은 `MCPClient` 클래스 (`skilljar/S6_03_mcp_client.ipynb`), 본 튜토리얼은 low-level부터 단계적으로 접근합니다.


## §0. 강의노트 매핑

| 본 튜토리얼 단계 | 강의노트 위치 | 비고 |
|------|-----|-----|
| §1 low-level API | `Week_07.md §2.1.1` | `ClientSession` 기초 |
| §2 도구 호출 | `Week_07.md §2.1.2` | `call_tool` |
| §3 Claude + MCP | `Week_07.md §2.1.3` | Tool Use 루프 결합 |
| §6 클래스 추상화 | `Week_07.md §2.1.4` | `MCPClient` 클래스 |


## §0. 환경 준비

import 에러가 발생하면 다음을 설치하세요:
```bash
pip install "mcp[cli]" anthropic python-dotenv
```

In [ ]:
# Setup
import asyncio
import json
import os
import anthropic
from dotenv import load_dotenv

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()  # .env 파일에서 ANTHROPIC_API_KEY 로드

MODEL = "claude-haiku-4-5"

# API 키 확인
api_key = os.getenv("ANTHROPIC_API_KEY")
if api_key:
    print(f"OK API 키 로드 완료 (마지막 4자: ...{api_key[-4:]})")
else:
    print("ANTHROPIC_API_KEY가 없습니다. .env 파일 또는 환경변수로 설정하세요.")

# tutorial_server.py 존재 확인
if os.path.exists("tutorial_server.py"):
    print("OK tutorial_server.py 존재 (T01에서 생성)")
else:
    print("tutorial_server.py가 없습니다. T01 §8을 먼저 실행하세요.")


## §1. low-level API — `ClientSession` + `stdio_client`

가장 원시적인 형태부터 시작합니다. 다음 구조를 이해하면 이후 모든 추상화의 의미가 명확해집니다.

```
StdioServerParameters  ← 서버 어떻게 띄울지 (command + args)
  ↓
stdio_client           ← 서버 프로세스 spawn + 표준 입출력 파이프
  ↓
ClientSession          ← JSON-RPC 세션 (initialize / list_tools / call_tool)
```

> 💡 **왜 stdio?** MCP는 stdio(표준 입출력) 또는 SSE(Server-Sent Events) 두 가지 transport를 지원합니다. 로컬 개발에서는 stdio가 가장 간단합니다.

In [ ]:
async def list_tools_basic():
    """가장 기본적인 형태로 서버에 연결하여 도구 목록 조회."""

    # Step 1: 서버 연결 파라미터
    params = StdioServerParameters(
        command="python",
        args=["tutorial_server.py"],
    )

    # Step 2: stdio 채널 열기 (서버 프로세스 spawn)
    async with stdio_client(params) as (read, write):
        # Step 3: 세션 생성
        async with ClientSession(read, write) as session:
            # Step 4: 핸드셰이크 (initialize)
            await session.initialize()

            # Step 5: 도구 목록 조회
            tools = await session.list_tools()

            print(f"사용 가능한 도구 ({len(tools.tools)}개):")
            for t in tools.tools:
                print(f"  - {t.name}: {t.description}")
            return tools


tools = await list_tools_basic()


> ☑ **체크포인트 1**: 3개 도구가 출력됐나요?
>
> 예상 출력:
> ```
> 사용 가능한 도구 (3개):
>   - get_current_time: 현재 날짜와 시간을 반환합니다.
>   - add_numbers: 두 숫자를 더합니다.
>   - calculate_area: 직사각형 면적을 계산합니다.
> ```


## §2. 도구 호출 — `call_tool`

도구 목록을 봤으니 이제 호출해 봅니다. `session.call_tool(name, arguments)`를 사용합니다.

In [ ]:
async def call_tools_demo():
    """서버에 연결하여 도구를 직접 호출."""
    params = StdioServerParameters(
        command="python",
        args=["tutorial_server.py"],
    )

    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # 면적 계산 호출
            result = await session.call_tool(
                "calculate_area",
                arguments={"width": 300, "height": 600, "unit": "mm"},
            )
            text = result.content[0].text if result.content else "(empty)"
            print(f"calculate_area(300, 600, mm) = {text}")

            # 시간 호출
            r2 = await session.call_tool(
                "get_current_time",
                arguments={"format": "%Y-%m-%d %H:%M"},
            )
            print(f"get_current_time = {r2.content[0].text}")


await call_tools_demo()


## §3. Claude API와 결합 — Tool Use 루프

이제 진짜 흥미로운 부분입니다. Claude API에 MCP 도구를 노출하면, **Claude가 스스로 어떤 도구를 호출할지 결정**합니다.

핵심 흐름:
1. MCP 서버에서 `list_tools()` → Claude Tool Use 형식으로 변환
2. Claude API 호출 (`messages.create(..., tools=...)`)
3. `stop_reason == "tool_use"`이면 도구 실행 → 결과 다시 Claude에 전달
4. `stop_reason == "end_turn"`이면 최종 답변 반환
5. 위 루프를 반복

> 💡 Week 04 Tool Use 루프와 **동일한 패턴**입니다. 다른 점은 도구 실행을 직접 하지 않고 **MCP 서버에 위임**한다는 것뿐.

In [ ]:
async def chat_with_mcp(user_message: str) -> str:
    """Claude + MCP 도구를 결합한 대화 함수."""
    api = anthropic.Anthropic()
    params = StdioServerParameters(
        command="python",
        args=["tutorial_server.py"],
    )

    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # 1) MCP tools를 Claude Tool Use 형식으로 변환
            mcp_tools = await session.list_tools()
            claude_tools = [
                {
                    "name": t.name,
                    "description": t.description,
                    "input_schema": t.inputSchema,
                }
                for t in mcp_tools.tools
            ]

            messages = [{"role": "user", "content": user_message}]
            print(f"[USER] {user_message}\n")

            # 2) Tool Use 루프
            while True:
                resp = api.messages.create(
                    model=MODEL,
                    max_tokens=1024,
                    tools=claude_tools,
                    messages=messages,
                )

                # 종료 조건: 최종 답변
                if resp.stop_reason == "end_turn":
                    final = "".join(
                        b.text for b in resp.content if hasattr(b, "text")
                    )
                    print(f"[CLAUDE] {final}")
                    return final

                # 도구 호출 요청
                elif resp.stop_reason == "tool_use":
                    messages.append({"role": "assistant", "content": resp.content})

                    tool_results = []
                    for block in resp.content:
                        if block.type == "tool_use":
                            print(f"  -> MCP 도구 호출: {block.name}({block.input})")

                            # 3) MCP 서버에 도구 실행 위임
                            result = await session.call_tool(
                                block.name, arguments=block.input
                            )
                            text = result.content[0].text if result.content else ""
                            tool_results.append({
                                "type": "tool_result",
                                "tool_use_id": block.id,
                                "content": text,
                            })
                            print(f"     <- 결과: {text}")

                    messages.append({"role": "user", "content": tool_results})

                else:
                    print(f"예상치 못한 stop_reason: {resp.stop_reason}")
                    break


# 시나리오 1: 시간 질의
answer = await chat_with_mcp("지금 몇 시야?")


## §4. 시나리오 2 — 면적 계산

이번엔 면적 계산 시나리오. Claude가 자연어를 파싱해서 `calculate_area` 도구를 자동으로 호출하는지 확인합니다.

In [ ]:
# 시나리오 2: 면적 계산 (자연어 → 도구 호출)
answer = await chat_with_mcp("가로 2.5m, 세로 3.2m 방의 면적은?")


> ☑ **체크포인트 2**: Claude가 `calculate_area` 도구를 호출하고, 결과를 한국어로 답했나요?
>
> 예상 흐름:
> ```
> [USER] 가로 2.5m, 세로 3.2m 방의 면적은?
>   -> MCP 도구 호출: calculate_area({'width': 2.5, 'height': 3.2, 'unit': 'm'})
>      <- 결과: 8.00 m²
> [CLAUDE] 가로 2.5m, 세로 3.2m 방의 면적은 8.00 m²입니다.
> ```
>
> ❌ 만약 도구 호출 없이 Claude가 직접 계산했다면, `claude_tools`가 제대로 전달됐는지 확인.


## §5. 추상화 단계 — 함수에서 클래스로

위 함수를 보면 매번 다음 작업을 반복합니다:

```python
async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        # ... 실제 작업
```

이 보일러플레이트(boilerplate)를 **클래스로 캡슐화**하면 호출부가 깔끔해집니다.

```python
async with SimpleMCPClient(...) as client:
    tools = await client.list_tools()
    result = await client.call_tool(...)
```

이는 Skilljar 원본의 `MCPClient` 클래스 설계와 동일한 사고 흐름입니다.


## §6. Skilljar 원본 `MCPClient` 클래스로의 연결

> 💡 **연결**: Skilljar 원본은 위 보일러플레이트를 `MCPClient` 클래스 (`__aenter__` / `__aexit__`)로 캡슐화합니다. 자세한 구현은 `skilljar/S6_03_mcp_client.ipynb`에서 확인하세요.
>
> 본 튜토리얼에서 low-level 흐름(`stdio_client` → `ClientSession` → `initialize`)을 이해했다면, `MCPClient` 클래스의 구현이 자연스럽게 이해됩니다. **클래스는 위 흐름을 그대로 캡슐화한 것일 뿐**입니다.

아래는 핵심만 추린 데모 클래스입니다.

In [ ]:
# 간단 데모: Skilljar 스타일 클래스 (요약 버전)
from contextlib import AsyncExitStack
from typing import Optional


class SimpleMCPClient:
    """Skilljar MCPClient의 학습용 축약 버전."""

    def __init__(self, command: str, args: list[str]):
        self._command = command
        self._args = args
        self._exit_stack = AsyncExitStack()
        self._session: Optional[ClientSession] = None

    async def __aenter__(self):
        # AsyncExitStack으로 stdio_client + ClientSession 컨텍스트 누적 관리
        params = StdioServerParameters(command=self._command, args=self._args)
        stdio = await self._exit_stack.enter_async_context(stdio_client(params))
        self._session = await self._exit_stack.enter_async_context(
            ClientSession(*stdio)
        )
        await self._session.initialize()
        return self

    async def __aexit__(self, exc_type, exc, tb):
        await self._exit_stack.aclose()
        self._session = None

    async def list_tools(self):
        return (await self._session.list_tools()).tools

    async def call_tool(self, name: str, args: dict):
        return await self._session.call_tool(name, arguments=args)


# 사용: 깔끔하게 한 줄로 컨텍스트 관리
async with SimpleMCPClient(command="python", args=["tutorial_server.py"]) as client:
    tools = await client.list_tools()
    print("도구 목록:")
    for t in tools:
        print(f"  - {t.name}")

    result = await client.call_tool(
        "add_numbers", {"a": 100, "b": 23}
    )
    print(f"\nadd_numbers(100, 23) = {result.content[0].text}")


> ☑ **체크포인트 3**: 클래스 버전이 동일하게 작동했나요?
>
> 예상 출력:
> ```
> 도구 목록:
>   - get_current_time
>   - add_numbers
>   - calculate_area
>
> add_numbers(100, 23) = 123.0
> ```
>
> 호출부가 훨씬 간결해진 것을 느끼셨나요? Skilljar `MCPClient` 클래스도 같은 원리입니다.


## §7. 트러블슈팅

| # | 에러 | 원인 | 해결 |
|---|-----|------|------|
| 1 | `FileNotFoundError: tutorial_server.py` | 서버 파일 없음 | T01 §8 다시 실행 |
| 2 | `anthropic.AuthenticationError` | API 키 문제 | `.env` 파일에 `ANTHROPIC_API_KEY=sk-...` 추가 |
| 3 | `RuntimeError: This event loop is already running` | Jupyter event loop 충돌 | `import nest_asyncio; nest_asyncio.apply()` |
| 4 | Claude가 도구를 호출하지 않음 | 모델이 자체 계산 선택 | 더 명시적인 프롬프트 (예: "도구를 사용하여 계산하세요") |
| 5 | `subprocess` 좀비 프로세스 | 비정상 종료 | 노트북 커널 재시작 + `pkill -f tutorial_server` |


## §8. 다음 단계

✅ T03 완료! 이제 다음을 할 수 있습니다:
- low-level `ClientSession` + `stdio_client`로 서버 연결
- `list_tools()` / `call_tool()` 직접 호출
- Claude API와 결합하여 자율 도구 호출 (Tool Use 루프 + MCP)
- 함수 → 클래스 추상화 (Skilljar `MCPClient`로 연결)

➡️ **다음 노트북** (T04~T06):
- `T04_resources_basics.ipynb` — `@mcp.resource()`로 정적/동적 데이터 노출 (재료 물성치 도메인)
- `T05_prompts_basics.ipynb` — `@mcp.prompt()`로 재사용 프롬프트 (KDS 검토)
- `T06_practice_review.ipynb` — 통합 실습 + 자가 점검

> 📚 **추가 학습**:
> - `skilljar/S6_03_mcp_client.ipynb` — Skilljar 원본 (`MCPClient` 풀 구현)
> - `Week_07.md §2.1` — 클라이언트 심화
